In [4]:
"""
Simple log generator:
Continuously emits synthetic logs to an ingestion endpoint.
Supports configurability through environment variables and CLI args.
"""

import os
import time
import random
import argparse
import requests
from datetime import datetime, timezone


DEFAULT_SERVICES = ['auth', 'payments', 'orders', 'search', 'worker']
DEFAULT_LEVELS = ['info', 'warning', 'error']
DEFAULT_MESSAGES = [
    'user login successful',
    'payment processed',
    'order created',
    'search index refreshed',
    'background job failed with exception',
    'db connection timeout',
    'cache miss for key',
]


def iso_timestamp():
    """Return UTC ISO timestamp including milliseconds."""
    return datetime.now(timezone.utc).isoformat(timespec='milliseconds')


def send_log(ingest_url, services, messages, levels, debug=False):
    payload = {
        'service': random.choice(services),
        'timestamp': iso_timestamp(),
        'level': random.choices(levels, weights=[80, 15, 5])[0],
        'message': random.choice(messages),
        'metadata': {'latency_ms': random.randint(10, 2000)}
    }

    try:
        resp = requests.post(ingest_url, json=payload, timeout=3)
        if debug:
            print(f"[DEBUG] Sent: {payload} | Status {resp.status_code}")
    except Exception as e:
        print(f"[ERROR] Failed to send log: {e}")


def main():
    parser = argparse.ArgumentParser(description="Synthetic log generator")
    parser.add_argument("--url", default=os.getenv("INGEST_URL", "http://localhost:9000/ingest"),
                        help="Ingestion URL")
    parser.add_argument("--rate", type=float, default=0.7,
                        help="Base interval between logs in seconds")
    parser.add_argument("--jitter", type=float, default=0.3,
                        help="Random jitter added/subtracted from rate")
    parser.add_argument("--debug", action="store_true",
                        help="Print debug logs")

    # Fix for Colab / Jupyter
    args, _ = parser.parse_known_args()

    print(f"Starting log generator → {args.url}")
    print(f"Rate: {args.rate}s ± {args.jitter}s\n")

    while True:
        send_log(
            ingest_url=args.url,
            services=DEFAULT_SERVICES,
            messages=DEFAULT_MESSAGES,
            levels=DEFAULT_LEVELS,
            debug=args.debug
        )

        sleep_time = max(0.1, args.rate + random.uniform(-args.jitter, args.jitter))
        time.sleep(sleep_time)



if __name__ == "__main__":
    main()


Starting log generator → http://localhost:9000/ingest
Rate: 0.7s ± 0.3s

[ERROR] Failed to send log: HTTPConnectionPool(host='localhost', port=9000): Max retries exceeded with url: /ingest (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5d7406da30>: Failed to establish a new connection: [Errno 111] Connection refused'))
[ERROR] Failed to send log: HTTPConnectionPool(host='localhost', port=9000): Max retries exceeded with url: /ingest (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5d7406c200>: Failed to establish a new connection: [Errno 111] Connection refused'))
[ERROR] Failed to send log: HTTPConnectionPool(host='localhost', port=9000): Max retries exceeded with url: /ingest (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b5d741c9ee0>: Failed to establish a new connection: [Errno 111] Connection refused'))
[ERROR] Failed to send log: HTTPConnectionPool(host='localhost', port=9000): Max ret

KeyboardInterrupt: 